# 5CCSAMLF CW2 Code Appendix
# Original TPCRP and Modified Centrality-Based TPCRP Implementations

In [1]:
import os
import sys
import numpy as np
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()

    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "src").is_dir():
            return base
        if (base / "typiclust-active-learning" / "src").is_dir():
            return base / "typiclust-active-learning"

    raise FileNotFoundError("Could not find project root containing 'src'.")

project_root = find_project_root(Path.cwd())
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Working directory:", Path.cwd())
print("Project root:", project_root)
print("Source path:", src_path)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import ToPILImage
import torchvision.models as models

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

Working directory: /Users/omxr/Desktop/VSCodeProjects/5CCSAMLF
Project root: /Users/omxr/Desktop/VSCodeProjects/5CCSAMLF/typiclust-active-learning
Source path: /Users/omxr/Desktop/VSCodeProjects/5CCSAMLF/typiclust-active-learning/src


## 1. SimCLR Augmentations 

In [2]:
class SimCLRTransform:
    def __init__(self, image_size: int = 32):
        color_jitter = transforms.ColorJitter(
            brightness=0.8,
            contrast=0.8,
            saturation=0.8,
            hue=0.2
        )

        self.base_transform = transforms.Compose([
            transforms.RandomResizedCrop(size=image_size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([color_jitter], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
        ])

    def __call__(self, x):
        x1 = self.base_transform(x)
        x2 = self.base_transform(x)
        return x1, x2

## 2. Contrastive loss

In [3]:
def nt_xent_loss(z1, z2, temperature: float = 0.5):
    batch_size = z1.size(0)

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    z = torch.cat([z1, z2], dim=0)  

    similarity = torch.matmul(z, z.T)  
    similarity = similarity / temperature

    mask = torch.eye(2 * batch_size, device=z.device).bool()
    similarity = similarity.masked_fill(mask, float("-inf"))

    targets = torch.arange(batch_size, device=z.device)
    targets = torch.cat([targets + batch_size, targets], dim=0)

    loss = F.cross_entropy(similarity, targets)
    return loss

## 3. SimCLR model 

In [4]:
class SimCLR(nn.Module):
    def __init__(self, projection_dim: int = 128):
        super().__init__()

        backbone = models.resnet18(weights=None)

        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        feature_dim = backbone.fc.in_features  # 512 for ResNet18

        self.projector = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)              
        h = torch.flatten(h, start_dim=1) 
        z = self.projector(h)            
        return h, z

## 4. SimCLR training 

In [5]:
from simclr.augmentations import SimCLRTransform
from simclr.simclr_model import SimCLR
from simclr.contrastive_loss import nt_xent_loss


class CIFAR10PairDataset:
    def __init__(self, root="./data", train=True, download=True, transform=None):
        self.dataset = datasets.CIFAR10(root=root, train=train, download=download)
        self.transform = transform
        self.to_pil = ToPILImage()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, _ = self.dataset[idx]

        if self.transform is None:
            raise ValueError("Transform must not be None for SimCLR training.")

        x1, x2 = self.transform(image)
        return x1, x2


def get_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def train_simclr(
    epochs: int = 50,
    batch_size: int = 256,
    lr: float = 1e-3,
    temperature: float = 0.5,
    save_path: str = "models/simclr_resnet18.pth"
):
    device = get_device()
    print("SimCLR training device:", device)

    transform = SimCLRTransform(image_size=32)
    dataset = CIFAR10PairDataset(transform=transform)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    model = SimCLR(projection_dim=128).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0

        for x1, x2 in loader:
            x1 = x1.to(device)
            x2 = x2.to(device)

            _, z1 = model(x1)
            _, z2 = model(x2)

            loss = nt_xent_loss(z1, z2, temperature=temperature)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(loader)
        print(f"Epoch {epoch + 1}/{epochs} | SimCLR Loss: {avg_loss:.4f}")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(model.state_dict(), save_path)
    print(f"Saved SimCLR model to {save_path}")

    return model

## 5. Embedding extraction

In [6]:
from simclr.simclr_model import SimCLR


def get_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def load_trained_simclr(model_path: str):
    device = get_device()
    model = SimCLR(projection_dim=128).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model, device


def get_cifar10_plain_loader(train=True, batch_size=256):
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    dataset = datasets.CIFAR10(
        root="./data",
        train=train,
        download=True,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    return dataset, loader


def extract_embeddings(model_path: str, train=True):
    model, device = load_trained_simclr(model_path)
    dataset, loader = get_cifar10_plain_loader(train=train)

    all_embeddings = []

    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            h, _ = model(images)
            h = F.normalize(h, dim=1)
            all_embeddings.append(h.cpu())

    embeddings = torch.cat(all_embeddings, dim=0).numpy()
    return dataset, embeddings

## 6. Clustering 

In [7]:

def cluster_features(features, num_clusters: int):
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(features)
    return cluster_labels

## 7. Typicality and Centrality 

In [8]:

def compute_typicality_of_points(cluster_features: np.ndarray, k: int = 20) -> np.ndarray:
    """
    Computes typicality for each point in a cluster
    typicality = 1 / average distance to k nearest neighbours
    """

    if len(cluster_features) == 1:
        return np.array([float("inf")])

    k = min(k, len(cluster_features) - 1)

    nbrs = NearestNeighbors(n_neighbors=k + 1)
    nbrs.fit(cluster_features)

    distances, _ = nbrs.kneighbors(cluster_features)

    neighbour_distances = distances[:, 1:]

    avg_distances = neighbour_distances.mean(axis=1)

    typicality = 1.0 / (avg_distances + 1e-8)

    return typicality


def compute_centrality(cluster_features: np.ndarray) -> np.ndarray:
    """
    Computes centrality = inverse distance to cluster centroid
    """

    centroid = np.mean(cluster_features, axis=0)

    distances = np.linalg.norm(cluster_features - centroid, axis=1)

    centrality = 1.0 / (distances + 1e-8)

    return centrality

## 8. Sample selection methods 

In [9]:
from typicality import compute_typicality_of_points,compute_centrality


def select_most_typical_per_cluster(
    features: np.ndarray,
    cluster_labels: np.ndarray,
    budget: int,
    min_cluster_size: int = 5
):
    selected_indices = []
    selected_set = set()

    unique_clusters = np.unique(cluster_labels)

    cluster_to_indices = {}
    for cluster_id in unique_clusters:
        indices = np.where(cluster_labels == cluster_id)[0]
        if len(indices) >= min_cluster_size:
            cluster_to_indices[cluster_id] = indices

    cluster_label_counts = {cluster_id: 0 for cluster_id in cluster_to_indices.keys()}

    while len(selected_indices) < budget and len(cluster_to_indices) > 0:
        min_count = min(cluster_label_counts.values())

        candidate_clusters = [
            cluster_id for cluster_id, count in cluster_label_counts.items()
            if count == min_count
        ]

        chosen_cluster = max(candidate_clusters, key=lambda cid: len(cluster_to_indices[cid]))
        cluster_indices = cluster_to_indices[chosen_cluster]

        available_indices = [idx for idx in cluster_indices if idx not in selected_set]

        if len(available_indices) < min_cluster_size:
            del cluster_to_indices[chosen_cluster]
            del cluster_label_counts[chosen_cluster]
            continue

        cluster_features = features[available_indices]
        k = min(20, len(available_indices) - 1)

        if k < 1:
            del cluster_to_indices[chosen_cluster]
            del cluster_label_counts[chosen_cluster]
            continue

        scores = compute_typicality_of_points(cluster_features, k=k)
        best_local_index = int(np.argmax(scores))
        best_global_index = available_indices[best_local_index]

        selected_indices.append(best_global_index)
        selected_set.add(best_global_index)
        cluster_label_counts[chosen_cluster] += 1

    return selected_indices


def select_weighted_typical_samples(
    features: np.ndarray,
    cluster_labels: np.ndarray,
    budget: int,
    min_cluster_size: int = 5
):
    """
    Modified TPCRP:
    allocate selections approximately proportional to cluster size,
    then pick the most typical points inside each cluster.
    """
    selected_indices = []

    unique_clusters = np.unique(cluster_labels)

    cluster_to_indices = {}
    cluster_sizes = {}
    for cluster_id in unique_clusters:
        indices = np.where(cluster_labels == cluster_id)[0]
        if len(indices) >= min_cluster_size:
            cluster_to_indices[cluster_id] = indices
            cluster_sizes[cluster_id] = len(indices)

    if not cluster_to_indices:
        return selected_indices

    total_points = sum(cluster_sizes.values())

    # Initial allocation proportional to cluster size
    allocation = {}
    remainders = []

    allocated_total = 0
    for cluster_id, size in cluster_sizes.items():
        exact_share = budget * (size / total_points)
        base_share = int(np.floor(exact_share))
        allocation[cluster_id] = base_share
        allocated_total += base_share
        remainders.append((cluster_id, exact_share - base_share))

    remaining_budget = budget - allocated_total
    remainders.sort(key=lambda x: x[1], reverse=True)

    for i in range(remaining_budget):
        cluster_id = remainders[i % len(remainders)][0]
        allocation[cluster_id] += 1

    for cluster_id in allocation:
        if allocation[cluster_id] == 0 and len(cluster_to_indices[cluster_id]) >= min_cluster_size:
            allocation[cluster_id] = 1

    while sum(allocation.values()) > budget:
        removable = [cid for cid, count in allocation.items() if count > 1]
        if not removable:
            break
        smallest = min(removable, key=lambda cid: cluster_sizes[cid])
        allocation[smallest] -= 1

    # Select top typical points within each cluster
    for cluster_id, num_to_select in allocation.items():
        if num_to_select <= 0:
            continue

        cluster_indices = cluster_to_indices[cluster_id]
        cluster_features = features[cluster_indices]

        k = min(20, len(cluster_indices) - 1)
        if k < 1:
            continue

        scores = compute_typicality_of_points(cluster_features, k=k)
        ranked_local_indices = np.argsort(scores)[::-1]  # descending

        take = min(num_to_select, len(ranked_local_indices))
        chosen_local = ranked_local_indices[:take]
        chosen_global = cluster_indices[chosen_local]

        selected_indices.extend(chosen_global.tolist())

    selected_indices = selected_indices[:budget]

    return selected_indices



def select_centrality_typical_samples(features, cluster_labels, budget, alpha=0.7):
    """
    Modified TPCRP:
    combine typicality + centrality score
    """

    selected_indices = []

    unique_clusters = np.unique(cluster_labels)

    for cluster_id in unique_clusters:

        cluster_indices = np.where(cluster_labels == cluster_id)[0]

        if len(cluster_indices) == 0:
            continue

        cluster_features = features[cluster_indices]

        typicality = compute_typicality_of_points(cluster_features)

        centrality = compute_centrality(cluster_features)

        # normalize scores
        typicality = (typicality - typicality.min()) / (typicality.max() - typicality.min() + 1e-8)
        centrality = (centrality - centrality.min()) / (centrality.max() - centrality.min() + 1e-8)

        score = alpha * typicality + (1 - alpha) * centrality

        best_index_local = np.argmax(score)

        best_index_global = cluster_indices[best_index_local]

        selected_indices.append(best_index_global)

    return selected_indices[:budget]

## 9. Random baseline 

In [10]:

def select_random_samples(dataset_size: int, budget: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    selected_indices = rng.choice(dataset_size, size=budget, replace=False)
    return selected_indices.tolist()

## 10. Test loader 

In [11]:

def get_cifar10_test_loader(batch_size: int = 256):
    transform = transforms.Compose([
        transforms.Resize((96, 96)),
        transforms.ToTensor(),
    ])

    dataset = datasets.CIFAR10(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return dataset, loader

## 11. Downstream classifier training 

In [12]:

def get_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def create_selected_subset(dataset, selected_indices):
    return Subset(dataset, selected_indices)


def create_model(num_classes: int = 10):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def train_model(train_dataset, test_loader, epochs: int = 5, batch_size: int = 32):
    device = get_device()
    print("Training device:", device)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model = create_model(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss = running_loss + loss.item()
            _, predicted = torch.max(outputs, 1)
            total = total + labels.size(0)
            correct = correct + (predicted == labels).sum().item()

        train_acc = 100.0 * correct / total if total > 0 else 0.0
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Loss: {running_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}%"
        )

    test_acc = evaluate_model(model, test_loader, device)
    return model, test_acc


def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100.0 * correct / total if total > 0 else 0.0
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

## 13. Original TPCRP pipeline 

In [13]:

SUBSET_SIZE = 50000
BUDGET = 200
EPOCHS = 30
SIMCLR_EPOCHS = 30
NUM_RUNS = 10 #5

MODEL_PATH = "models/simclr_resnet18.pth"


def run_tpcrp(dataset, features, test_loader):
    print("\n--- Running TPCRP ---")

    print("1. Clustering SimCLR embeddings...")
    cluster_labels = cluster_features(features, num_clusters=BUDGET)

    print("2. Selecting most typical sample per cluster...")
    selected_indices = select_most_typical_per_cluster(
        features,
        cluster_labels,
        budget=BUDGET
    )

    print(f"3. Number of selected samples: {len(selected_indices)}")

    selected_labels = [dataset[i][1] for i in selected_indices]
    print("4. Selected labels:", selected_labels)

    train_subset = create_selected_subset(dataset, selected_indices)

    print("5. Training classifier on TPCRP samples...")
    _, test_acc = train_model(train_subset, test_loader, EPOCHS, batch_size=16)

    return selected_indices, test_acc

## 14. Modified centrality based typicality TPCRP pipeline 

In [14]:
def run_centrality_tpcrp(dataset, features, test_loader):

    print("\n--- Running Centrality-Aware TPCRP ---")

    print("1. Clustering embeddings...")
    cluster_labels = cluster_features(features, num_clusters=BUDGET)

    print("2. Selecting centrality-weighted typical samples...")
    selected_indices = select_centrality_typical_samples(
        features,
        cluster_labels,
        budget=BUDGET
    )

    print(f"3. Number of selected samples: {len(selected_indices)}")

    selected_labels = [dataset[i][1] for i in selected_indices]
    print("4. Selected labels:", selected_labels)

    train_subset = create_selected_subset(dataset, selected_indices)

    print("5. Training classifier...")
    _, test_acc = train_model(train_subset, test_loader, EPOCHS, batch_size=16)

    return selected_indices, test_acc

## 14. Random baseline 


In [15]:
def run_random_baseline(dataset, test_loader):
    print("\n--- Running Random Baseline ---")

    selected_indices = select_random_samples(len(dataset), BUDGET, seed=42)

    print(f"1. Number of selected samples: {len(selected_indices)}")

    selected_labels = [dataset[i][1] for i in selected_indices]
    print("2. Selected labels:", selected_labels)

    train_subset = create_selected_subset(dataset, selected_indices)

    print("3. Training classifier on random samples...")
    _, test_acc = train_model(train_subset, test_loader, EPOCHS, batch_size=16)

    return selected_indices, test_acc

## 15. Main experiment execution 

In [16]:

_, test_loader = get_cifar10_test_loader()

results_dir = project_root / "results"
models_dir = project_root / "models"
results_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)

MODEL_PATH = project_root / "models" / "simclr_resnet18.pth"
print("Using model path:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

if not MODEL_PATH.exists():
    train_simclr(
        epochs=SIMCLR_EPOCHS,
        batch_size=128,
        lr=1e-3,
        temperature=0.5,
        save_path=str(MODEL_PATH)
    )

dataset, features = extract_embeddings(str(MODEL_PATH), train=True)
features = features[:SUBSET_SIZE]
dataset = Subset(dataset, list(range(SUBSET_SIZE)))

tpcrp_indices, tpcrp_acc = run_tpcrp(dataset, features, test_loader)
centrality_indices, centrality_acc = run_centrality_tpcrp(dataset, features, test_loader)
random_indices, random_acc = run_random_baseline(dataset, test_loader)


print("Original TPCRP Accuracy:", tpcrp_acc)
print("Modified Centrality TPCRP Accuracy:", centrality_acc)
print("Random baseline accuracy:", random_acc)

/Users/omxr/Desktop/VSCodeProjects/5CCSAMLF/typiclust-active-learning/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Using model path: /Users/omxr/Desktop/VSCodeProjects/5CCSAMLF/typiclust-active-learning/models/simclr_resnet18.pth
Model exists: True

--- Running TPCRP ---
1. Clustering SimCLR embeddings...
2. Selecting most typical sample per cluster...
3. Number of selected samples: 200
4. Selected labels: [6, 8, 7, 9, 9, 0, 8, 5, 6, 1, 7, 0, 7, 1, 9, 1, 6, 0, 6, 6, 7, 7, 1, 6, 8, 0, 4, 1, 4, 4, 3, 9, 1, 1, 1, 2, 9, 7, 9, 0, 3, 4, 4, 2, 8, 4, 1, 7, 6, 1, 8, 4, 5, 2, 1, 8, 1, 5, 1, 5, 5, 7, 5, 6, 6, 1, 1, 0, 8, 4, 8, 4, 2, 1, 7, 7, 0, 0, 9, 2, 6, 1, 7, 5, 2, 1, 5, 2, 3, 6, 1, 1, 8, 6, 4, 5, 7, 7, 2, 9, 2, 1, 5, 9, 7, 4, 1, 5, 4, 9, 0, 7, 3, 6, 2, 2, 6, 8, 1, 2, 6, 4, 4, 7, 6, 6, 1, 4, 9, 5, 5, 7, 9, 6, 2, 6, 1, 9, 6, 6, 3, 7, 4, 0, 6, 4, 4, 2, 6, 4, 4, 5, 9, 8, 5, 3, 6, 6, 7, 1, 4, 5, 8, 3, 3, 5, 1, 0, 0, 4, 7, 0, 2, 2, 5, 3, 4, 8, 2, 7, 2, 3, 8, 6, 4, 0, 4, 5, 8, 4, 9, 4, 0, 0, 4, 4, 0, 2, 1, 1]
5. Training classifier on TPCRP samples...
Training device: mps
Epoch 1/30 | Loss: 35.0689 | Train Acc: 